# Gate 3 evaluation and reproducibility evidence
This notebook validates and reproduces the tf-idf and keyword baseline results in [tf_idf_baseline](./tf_idf_baseline.ipynb) and [keyword_baseline](./keyword_baseline.ipynb) repsectively.

This notebook covers the Gate 3 items under **Evaluation and reproducibility evidence** and **Authorization to run the official-test evaluation**. It works on **validation data only**: it re-scores the validation chunks with both frozen baselines, checks the prediction and label contracts, recalculates the metrics, runs the locked Gate 4 rules (bootstrap, support warnings, error populations, error selection) on validation, and records the run receipt, artifact manifest and an authorization draft.

The official-test file is never parsed. Its bytes are hashed only to fill in the authorization's `official_source_sha256`.

Shared rules live in `evaluation.py`, so the official-test evaluation will import the same code. Its hash is recorded in the run receipt.

Some checklist items need people, :the independent recalculation by a second teammate (use `validation_rows.csv`), and signing the authorization.

In [41]:
import hashlib
import json
import os
import platform
import subprocess
import sys
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))  # the keyword baseline is the package src.keyword_baseline

import evaluation as ev
from text_normalization import normalize_text
from src.keyword_baseline.evaluate import max_possible_score
from src.keyword_baseline.normalize import normalize_with_map
from src.keyword_baseline.scorer import prepare_category, score_text

GATE3_DIR = PROJECT_ROOT / "data" / "gate3"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
KEYWORD_RULES = PROJECT_ROOT / "configs" / "keyword_rules_v1.json"
KEYWORD_THRESHOLDS = PROJECT_ROOT / "configs" / "keyword_thresholds_v1.json"
KEYWORD_REPORTS = PROJECT_ROOT / "reports" / "keyword_baseline"
TFIDF_REPORTS = PROJECT_ROOT / "reports" / "tf_idf_baseline"

CHUNKING_CONFIG = {"config_version": "gate3_chunking_v1", "token_pattern": r"\S+",
                   "chunk_size": 768, "overlap": 512, "step_size": 256}
EVAL_SET_ID = "validation__gate3_chunking_v1"  # stable id used in the error-selection ranking key
BASELINES = ["keyword", "tfidf"]

## Approved validation chunks and labels

Chunks come from the approved `gate3_chunking_v1` method (768-token chunks, 512-token overlap). Every chunk's text is checked against its `text_hash`, every chunk must carry the approved settings, and every contract must be a validation contract in the frozen split manifest. Context groups come from the manifest.

In [42]:
def load_chunks(path):
    '''Loads a chunk file and checks every chunk's text against its recorded text_hash.'''
    df = pd.read_csv(path)
    # same CRLF -> LF repair as tf_idf_baseline.load_chunks, for checkouts that converted line endings
    df["text"] = df["text"].str.replace("\r\n", "\n", regex=False)
    hashes = df["text"].map(lambda t: hashlib.sha256(t.encode("utf-8")).hexdigest())
    assert (hashes == df["text_hash"]).all(), f"{path.name} fails the text_hash check"
    return df

val_chunks = load_chunks(GATE3_DIR / "validation_chunks_768_512.csv")
val_labels = pd.read_csv(GATE3_DIR / "validation_chunk_labels_768_512.csv")
manifest = pd.read_csv(SPLITS_DIR / "split_manifest.csv")

assert val_chunks["chunk_id"].is_unique
assert (val_chunks["config_version"] == CHUNKING_CONFIG["config_version"]).all()
for col in ["chunk_size", "overlap", "step_size"]:
    assert (val_chunks[col] == CHUNKING_CONFIG[col]).all(), col
assert set(val_chunks["title"]) == set(manifest.loc[manifest["split"] == "validation", "title"])

CATEGORIES = sorted(val_labels["category"].unique())
assert len(CATEGORIES) == 10
chunk_title = val_chunks.set_index("chunk_id")["title"]
title_group = manifest.set_index("title")["context_group_id"]
print(f"{len(val_chunks):,} validation chunks from {val_chunks['title'].nunique()} contracts, "
      f"{val_chunks['title'].map(title_group).nunique()} context groups")

2,481 validation chunks from 82 contracts, 82 context groups


## Re-scoring validation with both frozen baselines

Both scorers are rebuilt from committed files alone, with no labels:

- **Keyword**: `configs/keyword_rules_v1.json` scored with `src.keyword_baseline.scorer`, thresholds from `configs/keyword_thresholds_v1.json`. The keyword normalizer must give exactly the same text as the shared `normalize_text`.
- **TF-IDF**: `selected_model.npz` (array hashes checked against `selected_model.json`), with `margin = tfidf(normalize_text(text)) @ coef + intercept`.

Every chunk gets a score for all 10 categories, masked or not. The rescored values are compared with the committed validation outputs: keyword scores must match exactly, and TF-IDF margins must be within the reproducibility tolerance (`atol=1e-10`, `rtol=1e-8`) with identical detections.

In [43]:
rules = json.loads(KEYWORD_RULES.read_text(encoding="utf-8"))
keyword_meta = json.loads(KEYWORD_THRESHOLDS.read_text(encoding="utf-8"))
assert keyword_meta["rule_set_version"] == rules["rule_set_version"]
prepared = {c: prepare_category(rules["categories"][c]) for c in CATEGORIES}

keyword_rows = []
for chunk_id, text in zip(val_chunks["chunk_id"], val_chunks["text"]):
    normalized = normalize_with_map(text)  # normalize once per chunk for all 10 categories
    assert normalized[0] == normalize_text(text), f"keyword and shared normalization differ on {chunk_id}"
    for c in CATEGORIES:
        keyword_rows.append({"chunk_id": chunk_id, "category": c,
                             "score": float(score_text(text, prepared[c], normalized=normalized)[0])})
keyword_pred = pd.DataFrame(keyword_rows)
keyword_pred["threshold"] = keyword_pred["category"].map(keyword_meta["thresholds"]).astype(float)

committed = pd.read_csv(KEYWORD_REPORTS / "validation_predictions.csv")
check = keyword_pred.merge(committed, on=["chunk_id", "category"], how="outer", validate="one_to_one")
assert (check["score_x"] == check["score_y"]).all(), "keyword scores differ from validation_predictions.csv"
print(f"Keyword: {len(keyword_pred):,} scores match the committed validation predictions exactly")

Keyword: 24,810 scores match the committed validation predictions exactly


In [44]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_meta = json.loads((TFIDF_REPORTS / "selected_model.json").read_text(encoding="utf-8"))
with np.load(TFIDF_REPORTS / "selected_model.npz", allow_pickle=False) as npz:
    model = {name: npz[name] for name in npz.files}

def array_sha256(a):
    '''Hash of an array's dtype, shape and bytes (same as tf_idf_baseline.ipynb).'''
    h = hashlib.sha256(f"{a.dtype.str}|{a.shape}|".encode())
    h.update(np.ascontiguousarray(a).tobytes())
    return h.hexdigest()

for name, a in model.items():
    assert array_sha256(a) == tfidf_meta["arrays"][name]["sha256"], f"{name} doesn't match its recorded hash"
assert list(model["categories"]) == CATEGORIES

params = {k: tuple(v) if k == "ngram_range" else v for k, v in tfidf_meta["vectorizer"].items()}
vectorizer = TfidfVectorizer(**params, vocabulary={str(t): j for j, t in enumerate(model["features"])})
vectorizer.idf_ = model["idf"]
margins = vectorizer.transform(val_chunks["text"].map(normalize_text)) @ model["coef"].T + model["intercept"]

tfidf_pred = pd.DataFrame({
    "chunk_id": np.repeat(val_chunks["chunk_id"].to_numpy(), len(CATEGORIES)),
    "category": np.tile(CATEGORIES, len(val_chunks)),
    "score": margins.ravel(),  # row-major: chunk by chunk, categories in model order
    "threshold": np.tile(model["threshold"], len(val_chunks)),
})

# committed margins cover the unmasked rows of every configuration; its header has a stale "x_config" name
committed = pd.read_csv(TFIDF_REPORTS / "validation_margins.csv").rename(columns={"x_config": "config"})
committed = committed[committed["config"] == tfidf_meta["selected_config"]]
check = committed.merge(tfidf_pred, on=["chunk_id", "category"], how="left", validate="one_to_one")
np.testing.assert_allclose(check["score"], check["margin"], rtol=1e-8, atol=1e-10)
assert ((check["score"] >= check["threshold"]) == (check["margin"] >= check["threshold"])).all()
print(f"TF-IDF: {len(check):,} unmasked margins reproduced within tolerance, detections identical")

TF-IDF: 24,646 unmasked margins reproduced within tolerance, detections identical


## Prediction contract

For each baseline: unique `(chunk_id, category)` keys, exactly 10 predictions for every approved chunk, finite scores, one threshold per category, and detections recalculated as `score >= threshold`. Both baselines must use the same record keys.

In [45]:
predictions = {}
for baseline, pred in [("keyword", keyword_pred), ("tfidf", tfidf_pred)]:
    pred = pred.assign(baseline=baseline, detected=pred["score"] >= pred["threshold"])
    ev.check_predictions(pred, val_chunks, CATEGORIES)
    predictions[baseline] = pred

key = lambda p: set(zip(p["chunk_id"], p["category"]))
assert key(predictions["keyword"]) == key(predictions["tfidf"]), "baselines use different record keys"
print("Both baselines pass the prediction contract and share the same", len(key(predictions["keyword"])), "keys")

Both baselines pass the prediction contract and share the same 24810 keys


## Joining labels

Predictions join to labels one-to-one; targets and masks must be binary. All rows, masked or not, go into `validation_rows.csv` (the frozen row-level results for independent recalculation). Only unmasked rows go into metrics, intervals and error lists.

In [46]:
rows = pd.concat([ev.join_labels(predictions[b], val_labels) for b in BASELINES], ignore_index=True)
rows["title"] = rows["chunk_id"].map(chunk_title)
rows["context_group_id"] = rows["title"].map(title_group)
assert rows["context_group_id"].notna().all()
rows = rows[["baseline", "chunk_id", "title", "context_group_id", "category",
             "score", "threshold", "detected", "target", "mask"]]

eval_rows = ev.unmasked(rows)
print(rows.groupby("baseline")["mask"].agg(rows="size", masked="sum"))

           rows  masked
baseline               
keyword   24810     164
tfidf     24810     164


## Reproducing Thresholds

Thresholds are re-selected from the rescored validation results with the Glossary rule (highest F1, then higher recall, then lower threshold) and must equal the frozen ones. Keyword candidates are the integers 1 to the category's maximum possible score; TF-IDF candidates are the distinct unmasked validation margins.

In [47]:
def select_threshold(scores, y, candidates):
    '''Highest F1, then higher recall, then lower threshold.'''
    detected = scores[:, None] >= candidates[None, :]
    tp = (detected & (y[:, None] == 1)).sum(axis=0)
    fp = (detected & (y[:, None] == 0)).sum(axis=0)
    _, recall, f1 = ev.prf(tp, fp, (y == 1).sum() - tp)
    return candidates[np.lexsort((candidates, -recall, -f1))[0]]  # last key sorts first

for c in CATEGORIES:
    kw = eval_rows[(eval_rows["baseline"] == "keyword") & (eval_rows["category"] == c)]
    candidates = np.arange(1, max_possible_score(rules["categories"][c]) + 1, dtype=float)
    assert select_threshold(kw["score"].to_numpy(), kw["target"].to_numpy(), candidates) == keyword_meta["thresholds"][c], c

    tf = eval_rows[(eval_rows["baseline"] == "tfidf") & (eval_rows["category"] == c)]
    chosen = select_threshold(tf["score"].to_numpy(), tf["target"].to_numpy(), np.unique(tf["score"]))
    np.testing.assert_allclose(chosen, tf["threshold"].iloc[0], rtol=1e-8, atol=1e-10)
print("Both baselines' thresholds reproduce for all 10 categories")

Both baselines' thresholds reproduce for all 10 categories


## Metrics

Per the metric calculation rules: per category `TP`, `FP`, `FN`, `TN`, positive and negative support, predicted positives, precision, recall and F1 (0 when a denominator is 0); macro-F1 is the plain mean of the 10 category F1s; micro metrics pool the counts. The same table carries the low-support warnings (`low_gold_support`, `low_predicted_support`, `zero_support`), with positive support in chunks, contracts and context groups.

The counts must match each baseline's own committed validation report.

In [48]:
assert [float(x) for x in ev.prf(0, 0, 0)] == [0.0, 0.0, 0.0]  # zero denominators return 0
assert [float(x) for x in ev.prf(0, 3, 0)] == [0.0, 0.0, 0.0]

tables = {b: ev.category_table(eval_rows[eval_rows["baseline"] == b]) for b in BASELINES}
category_results = pd.concat([t.assign(baseline=b) for b, t in tables.items()], ignore_index=True)
summaries = pd.DataFrame([{"baseline": b, **ev.summary(t)} for b, t in tables.items()])

# cross-check against the baselines' own reports
counts = ["tp", "fp", "fn", "tn"]
kw_report = pd.read_csv(KEYWORD_REPORTS / "validation_metrics.csv").set_index("category").sort_index()
assert (tables["keyword"].set_index("category")[counts] == kw_report[counts]).all().all()
tf_report = pd.read_csv(TFIDF_REPORTS / "category_results.csv")
tf_report = tf_report[tf_report["config"] == tfidf_meta["selected_config"]].set_index("category").sort_index()
assert (tables["tfidf"].set_index("category")[counts] == tf_report[counts]).all().all()

# evaluate.py averaged F1s already rounded to 4 places (0.6729 vs exact 0.67296), so allow that rounding
assert abs(summaries.set_index("baseline").loc["keyword", "macro_f1"] - keyword_meta["validation_macro_f1"]) <= 1e-4
tf_summary = pd.read_csv(TFIDF_REPORTS / "config_summary.csv").set_index("config").loc[tfidf_meta["selected_config"]]
assert np.isclose(summaries.set_index("baseline").loc["tfidf", "macro_f1"], tf_summary["macro_f1"])

summaries

,baseline,macro_f1,micro_precision,micro_recall,micro_f1,tp,fp,fn,tn
0,keyword,0.672964,0.717269,0.665425,0.690375,893,352,449,22952
1,tfidf,0.711664,0.762162,0.735469,0.748578,987,308,355,22996


In [49]:
category_results[["baseline", "category", "positive_support", "negative_support", "predicted_positive",
                  "precision", "recall", "f1", "positive_contracts", "positive_context_groups",
                  "predicted_context_groups", "low_gold_support", "low_predicted_support", "zero_support"]]

,baseline,category,positive_support,negative_support,predicted_positive,precision,recall,f1,positive_contracts,positive_context_groups,predicted_context_groups,low_gold_support,low_predicted_support,zero_support
0,keyword,Anti-Assignment,217,2231,213,0.793427,0.778802,0.786047,67,67,61,False,False,False
1,keyword,Cap On Liability,229,2217,157,0.878981,0.602620,0.715026,50,50,42,False,False,False
2,keyword,Change Of Control,73,2399,146,0.335616,0.671233,0.447489,19,19,35,True,False,False
3,keyword,Exclusivity,119,2337,141,0.524823,0.621849,0.569231,31,31,30,False,False,False
4,keyword,Expiration Date,192,2273,137,0.839416,0.598958,0.699088,71,71,46,False,False,False
5,keyword,License Grant,207,2251,172,0.843023,0.700483,0.765172,41,41,39,False,False,False
6,keyword,Non-Compete,72,2407,91,0.428571,0.541667,0.478528,19,19,23,True,False,False
7,keyword,Non-Transferable License,89,2387,80,0.962500,0.865169,0.911243,26,26,23,False,False,False
8,keyword,Renewal Term,78,2395,69,0.797101,0.705128,0.748299,31,31,26,False,False,False
9,keyword,Uncapped Liability,66,2407,39,0.820513,0.484848,0.609524,20,20,15,False,True,False


## Locked rules for the official evaluation, exercised on validation

These are implemented in `evaluation.py` and run here on validation, before any official-test label is used:

- **Paired uncertainty intervals**: 10,000 repetitions, seed `20260903`, resampling `context_group_id` with replacement; the same sample for both baselines; predictions and thresholds fixed; 95% percentile intervals for macro-F1, micro-F1 and each category's F1, plus TF-IDF minus keyword. A category is `unstable` if any repetition has no positive support.
- **Low-support warnings**: in the metrics table above.
- **Complete FP/FN populations** for both baselines and all 10 categories.
- **Repeatable error selection**: up to 5 per (baseline, category, error type) group, ranked by SHA-256 of seed `20260904`, eval-set id, baseline, category, error type, `context_group_id` and `chunk_id`; one per context group first, then fill. All 40 groups are listed, including empty ones.
- **Reviewer order**: seed `20260905`, two different reviewers per case.
- **Error-review categories**: `ev.PRIMARY_CAUSES` and `ev.SECONDARY_CAUSES`; the review sheet has one blank row per (case, reviewer).

The validation error sample doubles as the reviewers' practice set before they see official-test errors. `REVIEWERS` are placeholders: replace them with teammates' names before the practice round.

In [50]:
intervals = ev.bootstrap_intervals(eval_rows, BASELINES)
assert (intervals["lower_95"] <= intervals["upper_95"]).all()
assert ev.bootstrap_intervals(eval_rows, BASELINES, reps=200).equals(ev.bootstrap_intervals(eval_rows, BASELINES, reps=200))
intervals[intervals["category"] == ""]

,scope,metric,category,lower_95,upper_95,width,reps_without_positive_support,unstable
0,keyword,macro_f1,,0.626225,0.715029,0.088804,0,False
1,tfidf,macro_f1,,0.671976,0.744558,0.072582,0,False
2,tfidf_minus_keyword,macro_f1,,0.000399,0.073384,0.072985,0,False
3,keyword,micro_f1,,0.644478,0.733230,0.088752,0,False
4,tfidf,micro_f1,,0.717592,0.777739,0.060147,0,False
5,tfidf_minus_keyword,micro_f1,,0.024822,0.091333,0.066511,0,False


In [51]:
REVIEWERS = ["Annabel Wang", "reviewer_2"   , "reviewer_3", "reviewer_4"]  # placeholders: replace with teammates

errors = ev.error_population(eval_rows)
selected, error_groups = ev.select_errors(errors, EVAL_SET_ID, BASELINES, CATEGORIES)

assert len(error_groups) == 40
assert (error_groups["n_selected"] == error_groups["n_errors"].clip(upper=ev.ERRORS_PER_GROUP)).all()
for _, g in selected.groupby(["baseline", "category", "error_type"]):
    pool = errors[(errors["baseline"] == g["baseline"].iloc[0]) & (errors["category"] == g["category"].iloc[0])
                  & (errors["error_type"] == g["error_type"].iloc[0])]
    # a context group repeats only when the population has fewer than 5 distinct groups
    assert g["context_group_id"].nunique() == min(len(g), pool["context_group_id"].nunique())
assert selected.equals(ev.select_errors(errors, EVAL_SET_ID, BASELINES, CATEGORIES)[0])  # repeatable

review = ev.review_sheet(ev.assign_reviewers(selected, REVIEWERS))
assert (review.groupby("review_order")["reviewer"].nunique() == 2).all()
print(f"{len(errors):,} validation errors; {len(selected)} selected across 40 groups "
      f"({error_groups['no_errors'].sum()} with no errors); {len(review)} review rows")
error_groups.pivot_table(index="category", columns=["baseline", "error_type"], values="n_errors")

1,464 validation errors; 198 selected across 40 groups (0 with no errors); 396 review rows


baseline                        keyword                         tfidf  \
error_type               false_negative false_positive false_negative   
category                                                                
Anti-Assignment                    48.0           44.0           35.0   
Cap On Liability                   91.0           19.0           61.0   
Change Of Control                  24.0           97.0           35.0   
Exclusivity                        45.0           67.0           55.0   
Expiration Date                    77.0           22.0           45.0   
License Grant                      62.0           27.0           42.0   
Non-Compete                        33.0           52.0           33.0   
Non-Transferable License           12.0            3.0           13.0   
Renewal Term                       23.0           14.0           22.0   
Uncapped Liability                 34.0            7.0           14.0   

baseline                                 
error_type               false_positive  
category                                 
Anti-Assignment                    30.0  
Cap On Liability                   25.0  
Change Of Control                  35.0  
Exclusivity                        53.0  
Expiration Date                    25.0  
License Grant                      30.0  
Non-Compete                        42.0  
Non-Transferable License           28.0  
Renewal Term                        7.0  
Uncapped Liability                 33.0

## Saving results and the artifact manifest

Outputs go to `reports/gate3_evaluation/`, all prefixed `validation_` so they are never confused with official-test results. CSVs are written with `\n` line endings so their hashes match on any OS.

The manifest lists each file's hash, row count and columns. Each **Artifact Bundle ID** is a content hash over the bundle's files (path + file SHA-256), so the same files always give the same ID. Text files (code, JSON, Markdown) are hashed with LF line endings, matching the bytes Git stores; data CSVs are hashed as stored.

In [52]:
OUTPUT_DIR = PROJECT_ROOT / "reports" / "gate3_evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_RECEIPT = OUTPUT_DIR / "run_receipt.json"
ARTIFACT_MANIFEST = OUTPUT_DIR / "artifact_manifest.json"
AUTH_DRAFT_JSON = OUTPUT_DIR / "official_test_authorization_draft.json"
AUTH_DRAFT_MD = OUTPUT_DIR / "official_test_authorization_draft.md"
# the signed copies the team fills in (approver, executor, id, time) and commits
AUTH_JSON = PROJECT_ROOT / "docs" / "authorization" / "official_test_authorization.json"
AUTH_MD = PROJECT_ROOT / "docs" / "authorization" / "official_test_authorization.md"

outputs = {
    "validation_rows.csv": rows,
    "validation_category_results.csv": category_results,
    "validation_summary.csv": summaries,
    "validation_bootstrap_intervals.csv": intervals,
    "validation_error_population.csv": errors,
    "validation_error_groups.csv": error_groups,
    "validation_error_review_sheet.csv": review,
}
for name, df in outputs.items():
    df.to_csv(OUTPUT_DIR / name, index=False, lineterminator="\n")

def file_sha256(path):
    '''SHA-256 of a file; text files are hashed with LF line endings so Windows and Linux checkouts agree.'''
    data = Path(path).read_bytes()
    if Path(path).suffix in {".py", ".json", ".md", ".txt"}:
        data = data.replace(b"\r\n", b"\n")
    return hashlib.sha256(data).hexdigest()

def bundle_id(name, paths):
    '''Content-addressed bundle id: hash of the sorted (relative path, file hash) pairs.'''
    entries = sorted((Path(p).relative_to(PROJECT_ROOT).as_posix(), file_sha256(p)) for p in paths)
    return f"{name}-{hashlib.sha256(json.dumps(entries).encode()).hexdigest()[:16]}", entries

bundles = {
    "gate3_chunks": [GATE3_DIR / f"{s}_{k}_768_512.csv" for s in ["train", "validation"] for k in ["chunks", "chunk_labels"]],
    "keyword_baseline": [KEYWORD_RULES, KEYWORD_THRESHOLDS],
    # the NPZ file hash changes on every save; selected_model.json holds the stable per-array hashes
    "tfidf_baseline": [TFIDF_REPORTS / "selected_model.json"],
    "gate3_validation_evaluation": [OUTPUT_DIR / name for name in outputs],
}
bundle_ids = {}
bundle_records = {}
for name, paths in bundles.items():
    bundle_ids[name], entries = bundle_id(name, paths)
    bundle_records[bundle_ids[name]] = [{"path": p, "sha256": h} for p, h in entries]

artifact_manifest = {
    "bundle_id": bundle_ids["gate3_validation_evaluation"],
    "evaluation_set": EVAL_SET_ID,
    "files": [{"path": f"reports/gate3_evaluation/{name}", "sha256": file_sha256(OUTPUT_DIR / name),
               "rows": len(df), "columns": list(df.columns)} for name, df in outputs.items()],
    "sources": {k: v for k, v in bundle_records.items() if not k.startswith("gate3_validation_evaluation")},
}
ARTIFACT_MANIFEST.write_text(json.dumps(artifact_manifest, indent=2), encoding="utf-8")
bundle_ids

{'gate3_chunks': 'gate3_chunks-e0158a095e836390',
 'keyword_baseline': 'keyword_baseline-3bfd6d82e960fdac',
 'tfidf_baseline': 'tfidf_baseline-d830ae28570e4fdc',
 'gate3_validation_evaluation': 'gate3_validation_evaluation-2dfb1729d5a7f315'}

## Run receipt

The receipt records who ran this, when and where, and what went in:

- the code revision (and whether the working tree was clean);
- dependency versions and the dependency-lock hash;
- the notebook runtime;
- processing-thread settings;
- input hashes;
- the chunking record: criteria, candidate comparison, decision, approved settings, version and config hash;
- the locked-rule settings and `evaluation.py` hash;
- the separation-of-use check.

The **workflow hash** covers every workflow code and config file. Notebooks are hashed by their code cells only, so rerunning a notebook doesn't change it. The authorization must quote this hash.

The separation-of-use check confirms that no workflow file other than this notebook mentions the official-test files, and that every contract in this run's outputs is a validation contract. This notebook mentions them only to hash `test.json` for the authorization.

`code_revision.clean` is true when every workflow code file and input is committed. Files this run writes are ignored: `reports/gate3_evaluation/` and this notebook's cell outputs. A notebook counts as changed only if its code cells differ from `HEAD`. So: commit the code, run all cells, then commit the regenerated outputs. The receipt's `commit` is the code commit the outputs came from.

In [53]:
def git(*args):
    return subprocess.run(["git", *args], cwd=PROJECT_ROOT, capture_output=True, text=True, check=True).stdout.strip()

def code_text(path, raw=None):
    '''Workflow source: code cells for notebooks, the LF-normalized file for everything else.
    Pass raw bytes to read another version of the file (e.g. the committed one).'''
    raw = path.read_bytes() if raw is None else raw
    if path.suffix == ".ipynb":
        nb = json.loads(raw.decode("utf-8"))
        return "\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
    return raw.replace(b"\r\n", b"\n").decode("utf-8")

def notebook_code_changed(path):
    '''True if a notebook's code cells differ from HEAD; outputs and execution counts don't count.'''
    rel = path.relative_to(PROJECT_ROOT).as_posix()
    head = subprocess.run(["git", "show", f"HEAD:{rel}"], cwd=PROJECT_ROOT, capture_output=True)
    return head.returncode != 0 or code_text(path, head.stdout) != code_text(path)

THIS_NOTEBOOK = PROJECT_ROOT / "notebooks" / "gate3_evaluation.ipynb"
WORKFLOW_FILES = sorted(
    [PROJECT_ROOT / "notebooks" / f for f in ["create_split.ipynb", "review_bundles.ipynb", "gate3_chunking.ipynb",
                                              "tf_idf_baseline.ipynb", "keyword_baseline.ipynb", "gate3_evaluation.ipynb",
                                              "text_normalization.py", "evaluation.py"]]
    + sorted((PROJECT_ROOT / "src" / "keyword_baseline").glob("*.py"))
    + [KEYWORD_RULES, KEYWORD_THRESHOLDS, PROJECT_ROOT / "requirements.txt"])
workflow_entries = [(p.relative_to(PROJECT_ROOT).as_posix(), hashlib.sha256(code_text(p).encode()).hexdigest())
                    for p in WORKFLOW_FILES]
workflow_sha256 = hashlib.sha256(json.dumps(workflow_entries).encode()).hexdigest()
chunking_config_sha256 = hashlib.sha256(json.dumps(CHUNKING_CONFIG, sort_keys=True).encode()).hexdigest()

# separation of use: only this notebook may name the official-test files
mentions = [p.relative_to(PROJECT_ROOT).as_posix() for p in WORKFLOW_FILES
            if p != THIS_NOTEBOOK and ("test.json" in code_text(p) or "CUADv1" in code_text(p))]
assert not mentions, f"workflow files mention official-test files: {mentions}"
validation_titles = set(manifest.loc[manifest["split"] == "validation", "title"])
assert all(set(df["title"]) <= validation_titles for df in [rows, errors, review])

from threadpoolctl import threadpool_info

# "clean" means the code and inputs are committed. This run rewrites its own outputs (reports/gate3_evaluation/
# and this notebook's cell outputs), so those are ignored; notebooks count only if their code cells changed.
input_paths = [*bundles["gate3_chunks"], SPLITS_DIR / "split_manifest.csv", KEYWORD_RULES, KEYWORD_THRESHOLDS,
               KEYWORD_REPORTS / "validation_predictions.csv", TFIDF_REPORTS / "selected_model.json",
               TFIDF_REPORTS / "validation_margins.csv"]
tracked = [p.relative_to(PROJECT_ROOT).as_posix() for p in WORKFLOW_FILES + input_paths if p.suffix != ".ipynb"]
dirty = (git("status", "--porcelain", "--", *tracked).splitlines()
         + [f"code cells changed: {p.relative_to(PROJECT_ROOT).as_posix()}"
            for p in WORKFLOW_FILES if p.suffix == ".ipynb" and notebook_code_changed(p)])
run_receipt = {
    "workflow_run_id": f"gate3-validation-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}",
    "executed_by": git("config", "user.name"),
    "executed_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "where": {"platform": platform.platform(), "machine": platform.machine(), "python": sys.version},
    "code_revision": {"commit": git("rev-parse", "HEAD"), "branch": git("rev-parse", "--abbrev-ref", "HEAD"),
                      "clean": not dirty, "uncommitted": dirty},
    "notebook": THIS_NOTEBOOK.relative_to(PROJECT_ROOT).as_posix(),
    "runtime": {pkg: metadata.version(pkg) for pkg in
                ["numpy", "pandas", "scikit-learn", "scipy", "ipykernel", "jupyter_core", "nbformat"]},
    "dependency_lock": {"path": "requirements.txt", "sha256": file_sha256(PROJECT_ROOT / "requirements.txt")},
    "processing_threads": {
        "env": {k: os.environ.get(k) for k in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]},
        "cpu_count": os.cpu_count(),
        "threadpools": [{k: t[k] for k in ["internal_api", "num_threads", "version"]} for t in threadpool_info()],
    },
    "development_use_rules": [
        "official test set not loaded in development; model selection on data/splits train/validation only",
        "split frozen (seed 20260901); chunking frozen (gate3_chunking_v1)",
        "vectorizer, IDF and classifiers fitted on training chunks only",
        "keyword rules frozen before validation scoring (configs/keyword_freeze_v1.md)",
        "masked pairs excluded from thresholds, metrics and error lists",
    ],
    "inputs": {p.relative_to(PROJECT_ROOT).as_posix(): file_sha256(p) for p in input_paths},
    "tfidf_array_sha256": {k: v["sha256"] for k, v in tfidf_meta["arrays"].items()},
    "chunking": {
        "criteria": ["complete span coverage (no lost positive annotations)", "boundary-split spans",
                     "masked pairs", "overlap cost", "processing cost"],
        # from gate3_chunking.ipynb's comparison table (training spans, 10 core categories)
        "candidates": [
            {"method": "512/0", "total_chunks": 5280, "fully_covered": 2403, "boundary_split": 292, "lost_positives": 292, "masked_pairs": 391},
            {"method": "512/128", "total_chunks": 6875, "fully_covered": 2675, "boundary_split": 740, "lost_positives": 20, "masked_pairs": 495},
            {"method": "768/512", "total_chunks": 9765, "fully_covered": 2695, "boundary_split": 596, "lost_positives": 0, "masked_pairs": 699},
        ],
        "decision": "approved 768/512: the only candidate with zero lost positive annotations; 512/0 and 512/128 "
                    "rejected for lost positives (512/256 and 512/384 also still lost at least one)",
        "approved_settings": CHUNKING_CONFIG,
        "config_sha256": chunking_config_sha256,
        "source_notebook_sha256": hashlib.sha256(code_text(PROJECT_ROOT / "notebooks" / "gate3_chunking.ipynb").encode()).hexdigest(),
    },
    "locked_rules": {
        "evaluation_py_sha256": file_sha256(PROJECT_ROOT / "notebooks" / "evaluation.py"),
        "bootstrap": {"reps": ev.BOOTSTRAP_REPS, "seed": ev.BOOTSTRAP_SEED, "unit": "context_group_id",
                      "interval": "95% percentile", "paired": True},
        "low_support_context_groups": ev.LOW_SUPPORT_GROUPS,
        "error_selection": {"seed": ev.ERROR_SEED, "per_group": ev.ERRORS_PER_GROUP,
                            "protocol_version": ev.SAMPLING_PROTOCOL_VERSION,
                            "key": "sha256(seed|eval_set_id|baseline|category|error_type|context_group_id|chunk_id)"},
        "reviewer_seed": ev.REVIEWER_SEED,
        "primary_causes": ev.PRIMARY_CAUSES, "secondary_causes": ev.SECONDARY_CAUSES,
    },
    "workflow": {"sha256": workflow_sha256, "files": dict(workflow_entries)},
    "separation_of_use": {"workflow_files_mentioning_official_test": mentions,
                          "all_output_contracts_in_validation_split": True},
    "outputs": {"artifact_manifest": "reports/gate3_evaluation/artifact_manifest.json", "bundle_ids": bundle_ids},
}
RUN_RECEIPT.write_text(json.dumps(run_receipt, indent=2), encoding="utf-8")
if dirty:
    print("Not a clean rerun; commit these and rerun:", *dirty, sep="\n  ")
print("workflow_sha256:", workflow_sha256)

Not a clean rerun; commit these and rerun:
  code cells changed: notebooks/gate3_evaluation.ipynb
workflow_sha256: 5ea7a6498c459677a70005d2319eddd49839fde5d505d0cfa3c6bc5dd227a9c9


## Official-test authorization draft

The notebook pre-fills every field it can compute. For the official source file it hashes the bytes of `data/cuad/test.json` without parsing them. The team then completes the draft:

1. Copy both drafts to `docs/authorization/official_test_authorization.{json,md}`.
2. Fill in `approver`, `executor`, `authorization_id` and `authorization_time` identically in both files, and commit them.
3. Rerun this cell. It checks that the Markdown and JSON agree on every field, that no field is blank, and that the computed fields still match this run.

`approved_commit` must be the commit the official run will use. If the code changes after approval, the authorization is stale.

In [28]:
authorization = {
    "approved_commit": run_receipt["code_revision"]["commit"],
    "approver": None,
    "executor": None,
    "authorization_id": None,
    "workflow_sha256": workflow_sha256,
    "chunking_config_sha256": chunking_config_sha256,
    "artifact_bundle_ids": [bundle_ids[k] for k in ["gate3_chunks", "keyword_baseline", "tfidf_baseline",
                                                    "gate3_validation_evaluation"]],
    "official_source_sha256": hashlib.sha256((PROJECT_ROOT / "data" / "cuad" / "test.json").read_bytes()).hexdigest(),
    "authorization_time": None,
    "permitted_action": "one official-test evaluation of keyword_rules_v1 and the selected TF-IDF model on "
                        "data/cuad/test.json chunked with gate3_chunking_v1, using notebooks/evaluation.py",
}
assert list(authorization) == ev.AUTHORIZATION_FIELDS
AUTH_DRAFT_JSON.write_text(json.dumps(authorization, indent=2), encoding="utf-8")
AUTH_DRAFT_MD.write_text(
    "# Official-test evaluation authorization\n\n"
    + "".join(f"- {k}: {', '.join(v) if isinstance(v, list) else (v or '')}\n" for k, v in authorization.items()),
    encoding="utf-8")

if AUTH_JSON.exists() and AUTH_MD.exists():
    signed = json.loads(AUTH_JSON.read_text(encoding="utf-8"))
    ev.check_authorization(AUTH_MD, signed)
    stale = [k for k, v in authorization.items() if v is not None and signed[k] != v]
    assert not stale, f"authorization no longer matches this run: {stale}"
    print(f"Authorization {signed['authorization_id']} verified: Markdown and JSON agree")
else:
    print("Not yet authorized: drafts written to", AUTH_DRAFT_JSON.parent.relative_to(PROJECT_ROOT).as_posix())

Not yet authorized: drafts written to reports/gate3_evaluation
